<a href="https://colab.research.google.com/github/lynchronald1977-dotcom/agents-in-sdlc/blob/main/Yue2_text2music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <h1 align="center">🎵 YuE 2.0 (YuE2) Text-to-Music </h1>
<p align="center">
  <img src="https://img.shields.io/badge/Model-YuE2--3B--FP16-ff69b4?style=for-the-badge&logo=huggingface" />
  <img src="https://img.shields.io/badge/Backend-ComfyUI%20Native-blue?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Hardware-T4%20%7C%20A100%20%7C%20L4%20GPU-green?style=for-the-badge&logo=nvidia" />
</p>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

###

---
### ⚡ Quickstart Guide
1. **Runtime Verification**: Ensure **Runtime** &rarr; **Change runtime type** &rarr; **T4 GPU** (or better) is active.
2. **Run Step 1 to 3**: Install the environment, download the checkpoint, and launch the backend daemon.
3. **Use the Interactive Studio (Step 4)**: Customize prompts, style tags, lyrics, durations, seeds, and samplers using dropdowns and sliders, then generate and stream audio directly in the cell.

In [1]:
# @title 1. Hardware Verification & GPU Diagnostics
# @markdown Checks GPU memory and CUDA capabilities.
import torch
import sys
from IPython.display import HTML, display

if not torch.cuda.is_available():
    display(HTML("<div style='background-color:#ffebee;padding:12px;border-radius:8px;border-left:5px solid #f44336;'><h3 style='color:#c62828;margin:0;'>⚠️ GPU Not Detected!</h3><p style='margin:5px 0 0 0;'>Please navigate to <b>Runtime &rarr; Change runtime type</b> and select <b>T4 GPU</b> or higher.</p></div>"))
    raise SystemExit("GPU accelerator missing.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
cuda_ver = torch.version.cuda

display(HTML(f"""
<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
  <h3 style='color:#2e7d32;margin:0 0 8px 0;'>✅ Hardware Accelerator Ready</h3>
  <table style='font-family:monospace;font-size:13px;'>
    <tr><td><b>GPU Device:</b></td><td>&nbsp;{gpu_name}</td></tr>
    <tr><td><b>Total VRAM:</b></td><td>&nbsp;{vram_gb} GB</td></tr>
    <tr><td><b>CUDA Version:</b></td><td>&nbsp;{cuda_ver}</td></tr>
    <tr><td><b>PyTorch:</b></td><td>&nbsp;{torch.__version__}</td></tr>
  </table>
</div>
"""))

GPU Device:,Tesla T4
Total VRAM:,14.56 GB
CUDA Version:,12.8
PyTorch:,2.11.0+cu128


In [2]:
# @title 2. Install ComfyUI Core & Audio Dependencies
# @markdown Clones ComfyUI, installs audio libraries, and adds WebSocket client support for real-time telemetry.
import os
from IPython.display import HTML, display

print("🚀 Setting up ComfyUI audio environment...")

if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio accelerate sentencepiece mutagen websocket-client safetensors
!apt-get -y install -qq aria2 ffmpeg > /dev/null 2>&1

display(HTML("<div style='background-color:#e3f2fd;padding:12px;border-radius:8px;border-left:5px solid #2196f3;'><b style='color:#1565c0;'>✅ Environment installed and ready!</b></div>"))

🚀 Setting up ComfyUI audio environment...
Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 53340, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 53340 (delta 64), reused 38 (delta 38), pack-reused 53256 (from 3)
Receiving objects: 100% (53340/53340), 92.25 MiB | 18.13 MiB/s, done.
Resolving deltas: 100% (36196/36196), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.2/25.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [13]:
# @title 3. Download FP16 Model & Launch Fast Daemon
# @markdown Downloads pre-converted `yue2_3b_fp16.safetensors` directly from Jokality/YuE2-fp16 and starts ComfyUI.
import os
import time
import subprocess
import urllib.request
from IPython.display import HTML, display

%cd /content/ComfyUI

model_dir = "/content/ComfyUI/models/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "yue2_3b_fp16.safetensors")

# Download pre-converted FP16 Checkpoint directly (no conversion needed)
if not os.path.exists(model_path) or os.path.getsize(model_path) < 6_000_000_000:
    print("📥 Downloading YuE2 FP16 checkpoint (~7.26 GB) from Jokality/YuE2-fp16... Please wait.")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Jokality/YuE2-fp16/resolve/main/checkpoints/yue2_3b_fp16.safetensors" \
      -d "{model_dir}" \
      -o "yue2_3b_fp16.safetensors"
else:
    print("⚡ Found existing FP16 checkpoint in cache.")

# Terminate previous instances to avoid port conflicts
!pkill -9 -f "main.py" || true
time.sleep(2)

print("🔄 Starting ComfyUI Turbo Daemon with native FP16 acceleration...")
server_log = open("/content/comfy_server.log", "w")

server_cmd = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--force-fp16",
    "--highvram",
    "--dont-upcast-attention"
]

proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/content/ComfyUI")

server_alive = False
for i in range(40):
    if proc.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_alive = True
        break
    except Exception:
        time.sleep(1.5)

if server_alive:
    display(HTML("""
    <div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
      <b style='color:#2e7d32;'>🟢 YuE2 Native FP16 Engine Online!</b><br>
      <span style='color:#555;'>Model: <code>yue2_3b_fp16.safetensors</code> &bull; Hardware Tensor Cores Active. Run <b>Cell 4</b>.</span>
    </div>
    """))
else:
    print("\n❌ Startup failed. Log contents (/content/comfy_server.log):\n" + "-"*60)
    with open("/content/comfy_server.log", "r") as f:
        print(f.read())
    print("-" * 60)

/content/ComfyUI
⚡ Found existing FP16 checkpoint in cache.
^C
🔄 Starting ComfyUI Turbo Daemon with native FP16 acceleration...


In [6]:
# @title 5. File Manager: Download All Generated Tracks
# @markdown Compresses all audio files in the output directory into a single `.zip` and prompts a browser download.
import glob
import os
import shutil
from google.colab import files

output_dir = "/content/ComfyUI/output/audio"
all_audio = glob.glob(f"{output_dir}/*.*")

if not all_audio:
    print("No audio tracks generated yet.")
else:
    print(f"Found {len(all_audio)} track(s):")
    for track in all_audio:
        size_mb = round(os.path.getsize(track) / (1024*1024), 2)
        print(f" - {os.path.basename(track)} ({size_mb} MB)")

    zip_name = "/content/YuE2_Generated_Tracks"
    shutil.make_archive(zip_name, 'zip', output_dir)
    print("\n📦 Downloading tracks zip to your local computer...")
    files.download(f"{zip_name}.zip")

Found 1 track(s):
 - YuE2_Studio_00001.mp3 (4.44 MB)

📦 Downloading tracks zip to your local computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New Section